#### Įtartinų vartotojų profilio interpretacija

In [33]:
import pandas as pd
import numpy as np

In [34]:
# ─────────────────────────────────────────────────────────────────────────────
# 0. DUOMENŲ ĮKĖLIMAS IR GRUPIŲ SUDARYMAS
# ─────────────────────────────────────────────────────────────────────────────
features = pd.read_excel("Outputs/final_features.xlsx")
iforest_risk = pd.read_excel("Outputs/iforest_suspicious_risk_groups.xlsx")
lof_risk = pd.read_excel("Outputs/lof_suspicious_risk_groups.xlsx")
common = pd.read_excel("Outputs/common_risks.xlsx")   # 10 bendrų anomalijų

# Suformuojame keturias grupes
population   = features.copy()
iforest_feat = features[features["account_number"].isin(iforest_risk["account_number"])]
lof_feat     = features[features["account_number"].isin(lof_risk["account_number"])]
common_feat  = features[features["account_number"].isin(common["account_number"])]

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. LENTELĖ – CDR požymių medianų palyginimas (4 pagrindinės grupės)
# ─────────────────────────────────────────────────────────────────────────────

# Informatyviausi požymiai, kurie bus lyginami 
comparison_cols = [
    "total_calls",
    "mean_call_duration",
    "unique_contacts_called",
    "unique_numbers_per_account",
    "active_day_ratio",
    "avg_calls_per_active_day",
    "avg_calls_per_own_number",
    "night_call_share",
    "max_calls_to_single_contact",
    "mean_calls_to_single_contact",
    "failed_call_share",
    "busy_calls_count",
    "no_answer_calls_count",
]

# Palyginamoji lentelė su vidurkiais ir nukrypimu nuo populiacijos
comparison_table = pd.DataFrame({
    "Populiacija (n=9603)":      population[comparison_cols].mean(),
    "iForest (n=97)":            iforest_feat[comparison_cols].mean(),
    "LOF (n=97)":                lof_feat[comparison_cols].mean(),
    "Bendros anomalijos (n=10)": common_feat[comparison_cols].mean(),
}).round(2)

print("LENTELĖ: CDR požymių vidurkiai pagal grupę")
print(comparison_table.to_string())
print()


comparison_table["iForest nukrypimas (%)"] = (
    (comparison_table["iForest (n=97)"] - comparison_table["Populiacija (n=9603)"])
    / comparison_table["Populiacija (n=9603)"] * 100
).round(1)

comparison_table["LOF nukrypimas (%)"] = (
    (comparison_table["LOF (n=97)"] - comparison_table["Populiacija (n=9603)"])
    / comparison_table["Populiacija (n=9603)"] * 100
).round(1)

# Top 5 labiausiai besiskiriantys požymiai
print("Top 5 iForest vs. populiacija (pagal absoliutų nukrypimą %):")
print(comparison_table["iForest nukrypimas (%)"].abs().nlargest(5).to_string())
print()
print("Top 5 LOF vs. populiacija (pagal absoliutų nukrypimą %):")
print(comparison_table["LOF nukrypimas (%)"].abs().nlargest(5).to_string())
print()



LENTELĖ: CDR požymių vidurkiai pagal grupę
                              Populiacija (n=9603)  iForest (n=97)  LOF (n=97)  Bendros anomalijos (n=10)
total_calls                                 185.25         2808.51      899.95                    8247.70
mean_call_duration                            6.98            5.90        2.80                       0.80
unique_contacts_called                       18.80           99.31       32.52                     245.60
unique_numbers_per_account                    1.24            1.68        6.27                       4.90
active_day_ratio                              0.24            0.77        0.96                       0.65
avg_calls_per_active_day                      2.37            9.84       10.65                      55.29
avg_calls_per_own_number                    172.43         2248.17      444.40                    4111.72
night_call_share                              0.03            0.06        0.17                       0.11
max

In [36]:
# =============================================================================
# BENDRŲ ANOMALIJŲ (n=10) PROFILIS
# =============================================================================

# CDR profilis – bendros anomalijos vs. populiacija
profile_comparison = pd.DataFrame({
    "Populiacija (vidurkis)":       population[comparison_cols].mean(),
    "Bendros anomalijos (vidurkis)": common_feat[comparison_cols].mean(),
}).round(2)

profile_comparison["Nukrypimas (%)"] = (
    (profile_comparison["Bendros anomalijos (vidurkis)"]
     - profile_comparison["Populiacija (vidurkis)"])
    / profile_comparison["Populiacija (vidurkis)"] * 100
).round(1)

print("LENTELĖ: Bendrų anomalijų CDR profilis vs. populiacija")
print(profile_comparison.to_string())
print()

# Rizikos grupės kiekvienam iš 10 bendrų anomalijų vartotojų
print("Bendrų anomalijų rizikos grupės pagal abu metodus:")
print(common[["account_number", "lof_risk", "iforest_risk"]].to_string(index=False))
print()

# Aukštos rizikos vartotojai – patvirtinti abiem metodais
high_risk_both = common[
    (common["iforest_risk"] == "Aukšta rizika") &
    (common["lof_risk"]     == "Aukšta rizika")
]
print(f"Vartotojų, kuriems ABU metodai priskiria aukštą riziką: {len(high_risk_both)}")

if len(high_risk_both) > 0:
    hr_feat = features[features["account_number"].isin(
        high_risk_both["account_number"])][comparison_cols].mean().round(2)
    print("Aukštos rizikos vartotojų CDR profilis (vidurkiai):")
    print(hr_feat.to_string())
print()


LENTELĖ: Bendrų anomalijų CDR profilis vs. populiacija
                              Populiacija (vidurkis)  Bendros anomalijos (vidurkis)  Nukrypimas (%)
total_calls                                   185.25                        8247.70          4352.2
mean_call_duration                              6.98                           0.80           -88.5
unique_contacts_called                         18.80                         245.60          1206.4
unique_numbers_per_account                      1.24                           4.90           295.2
active_day_ratio                                0.24                           0.65           170.8
avg_calls_per_active_day                        2.37                          55.29          2232.9
avg_calls_per_own_number                      172.43                        4111.72          2284.6
night_call_share                                0.03                           0.11           266.7
max_calls_to_single_contact                  